# Carers Directory Scraper

Extracts organization names, emails, and URLs from carers.hk.

In [ ]:
import requests
from bs4 import BeautifulSoup
import re
import time
import pandas as pd
from google.colab import files

# --- Cloudflare Email Decoder ---
def decode_cf_email(encoded):
    try:
        email = ""
        key = int(encoded[:2], 16)
        for i in range(2, len(encoded), 2):
            email += chr(int(encoded[i:i+2], 16) ^ key)
        return email
    except:
        return None

# --- Configuration ---
base_url = 'https://www.carers.hk'
list_url = base_url + '/en-us/articles/service-and-resource'
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
}

results = []
test_limit = 382  # Number of pages to scrape (382 for full list)

print(f"🔍 Starting scrape for {test_limit} pages...")

for p in range(1, test_limit + 1):
    url = list_url + (f'?p={p}' if p > 1 else '')
    print(f"Processing Page {p}...")

    try:
        response = requests.get(url, headers=headers, timeout=15)
        soup = BeautifulSoup(response.text, 'html.parser')
        titles = soup.find_all('h2', class_='card-title fs20')

        for title in titles:
            link_tag = title.find('a')
            if not link_tag:
                continue

            org_name = link_tag.get_text(strip=True)
            unit_url = base_url + link_tag.get('href')

            try:
                u_resp = requests.get(unit_url, headers=headers, timeout=10)
                u_soup = BeautifulSoup(u_resp.text, 'html.parser')
                found_emails = set()

                # Decode Cloudflare protected emails
                cf_elements = u_soup.find_all(attrs={"data-cfemail": True})
                for el in cf_elements:
                    decoded = decode_cf_email(el.get('data-cfemail'))
                    if decoded:
                        found_emails.add(decoded)

                # Search by label
                labels = u_soup.find_all(string=re.compile(r'電郵|Email', re.I))
                for label in labels:
                    parent = label.find_parent()
                    if parent:
                        matches = re.findall(r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}', parent.get_text())
                        for m in matches:
                            found_emails.add(m)

                results.append({
                    'Page': p,
                    'Organization': org_name,
                    'Email Address': ", ".join(found_emails) if found_emails else "Not Found",
                    'Source URL': unit_url
                })
                time.sleep(0.4)

            except:
                continue

    except Exception as e:
        print(f"Error on page {p}: {e}")

# --- Save and Download ---
df = pd.DataFrame(results)
display(df)

if not df.empty:
    filename = 'carers_directory_full.csv'
    df.to_csv(filename, index=False, encoding='utf-8-sig')
    print(f"\n Scraping complete. Downloading {filename}...")
    files.download(filename)
else:
    print("\n No data found.")